<a href="https://colab.research.google.com/github/tahitiproo/Machine-Learning-RAG/blob/main/AlfaHackaton.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip uninstall -y faiss faiss-cpu faiss-gpu faiss-gpu-cu12 -q
!pip install -q "faiss-cpu==1.12.0" "numpy>=2.0.0" "sentence-transformers==3.1.1" "pandas>=2.2.2" "rank-bm25>=0.2.2"


import IPython
IPython.Application.instance().kernel.do_shutdown(True)  # перезапуск ядра


{'status': 'ok', 'restart': True}

In [ ]:
import numpy as np, pandas as pd, faiss
from sentence_transformers import SentenceTransformer
print("NumPy:", np.__version__)
print("FAISS:", faiss.__version__)


/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


NumPy: 2.0.2
FAISS: 1.12.0


In [ ]:
CHUNKS_CSV = "/content/chunked_websites_better.csv"
QUESTIONS_CSV = "/content/questions.csv"

In [ ]:
# Улучшенное чанкование: абзацы → предложения → пакуем в окна по словам, с перекрытием.
# Плюс: удаляем повторяющиеся строки внутри одной страницы, приклеиваем title/h1 к каждому чанку.

import re, pandas as pd, numpy as np
from typing import List

def clean_text_soft(t: str) -> str:
    if pd.isna(t): return ""
    t = str(t)
    # убираем теги, управляющие и лишние пробелы
    t = re.sub(r"<[^>]+>", " ", t)
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\s*\n\s*", "\n", t)
    # оставим символы валют/процентов/слеш, числа не трогаем
    t = t.replace("\xa0", " ").strip()
    return t

def split_paragraphs(t: str) -> List[str]:
    # делим по пустым строкам / длинным пробелам
    parts = re.split(r"\n\s*\n|(?:\r?\n){2,}", t)
    parts = [p.strip() for p in parts if p and p.strip()]
    return parts

def sent_split_ru(p: str) -> List[str]:
    # быстрый сплит по знакам конца предложения
    sents = re.split(r"(?<=[\.\!\?])\s+", p.strip())
    return [s for s in sents if s]

def dedup_lines_within_page(t: str, min_len=20, max_repeats=2) -> str:
    # убираем повторяющиеся короткие строки (меню/футер) > max_repeats раз в пределах страницы
    lines = [ln.strip() for ln in t.split("\n")]
    freq = {}
    out = []
    for ln in lines:
        key = ln.lower()
        if len(ln) >= min_len:
            freq[key] = freq.get(key, 0) + 1
            if freq[key] <= max_repeats:
                out.append(ln)
        else:
            out.append(ln)
    return "\n".join(out)

def pack_sentences(sents: List[str], target_words=160, overlap_sents=2) -> List[str]:
    # собираем чанки примерно по ~160 слов (~800-1000 символов) с перекрытием по предложениям
    chunks, cur, cur_w = [], [], 0
    for s in sents:
        w = len(s.split())
        if cur_w + w <= target_words or not cur:
            cur.append(s); cur_w += w
        else:
            chunks.append(" ".join(cur))
            # перекрытие
            cur = (cur[-overlap_sents:] if overlap_sents > 0 else []) + [s]
            cur_w = sum(len(x.split()) for x in cur)
    if cur: chunks.append(" ".join(cur))
    return chunks

def re_chunk_websites(input_csv, output_csv,
                      text_col="text",
                      id_col="web_id",
                      title_cols=("title","h1","article_title"),
                      target_words=160,
                      overlap_sents=2):
    df = pd.read_csv(input_csv)
    assert id_col in df.columns, f"нет колонки {id_col}"
    assert text_col in df.columns, f"нет колонки {text_col}"

    all_chunks = []
    for _, row in df.iterrows():
        web_id = int(row[id_col])
        raw = clean_text_soft(str(row[text_col]))
        raw = dedup_lines_within_page(raw)

        # сделаем префикс из заголовков, если есть
        headers = []
        for c in title_cols:
            if c in df.columns and pd.notna(row[c]) and str(row[c]).strip():
                headers.append(str(row[c]).strip())
        prefix = " — ".join(dict.fromkeys(headers))  # без дублей, сохраняя порядок
        prefix = prefix.strip()

        # абзацы → предложения → окна
        paragraphs = split_paragraphs(raw) if "\n" in raw else [raw]
        sents = []
        for p in paragraphs:
            sents.extend(sent_split_ru(p))

        chunks = pack_sentences(sents, target_words=target_words, overlap_sents=overlap_sents)

        for i, ch in enumerate(chunks):
            chunk_text = (prefix + ". " + ch) if prefix else ch
            all_chunks.append({
                "web_id": web_id,
                "chunk_id": f"{web_id}_{i}",
                "chunk_number": i+1,
                "chunk_text": chunk_text,
                "chunk_length_words": len(chunk_text.split())
            })

    out_df = pd.DataFrame(all_chunks)
    out_df.to_csv(output_csv, index=False, encoding="utf-8")
    print(f"Готово: {len(out_df)} чанков → {output_csv}")
    return out_df

WEBSITES_CSV = "/content/websites.csv"
chunked_better = re_chunk_websites(WEBSITES_CSV, "/content/chunked_websites_better.csv",
                                    text_col="text", id_col="web_id",
                                    target_words=160, overlap_sents=2)


In [ ]:
import re, pickle, numpy as np, pandas as pd, faiss
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

# ---- лёгкий препроцесс вопросов ----
EMOJI_RE = re.compile(
    "["                                 # emoji ranges
    "\U0001F600-\U0001F64F"
    "\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF"
    "\U0001F1E0-\U0001F1FF"
    "\U00002700-\U000027BF"
    "\U0001F900-\U0001F9FF"
    "\U00002600-\U000026FF"
    "\U00002B00-\U00002BFF"
    "\U0001FA70-\U0001FAFF"
    "]+", flags=re.UNICODE
)
EXTRA_SYMBOLS_RE = re.compile(r"[\u200d\u200c\uFE0E\uFE0F\U0001F3FB-\U0001F3FF]")

def preprocess_query(q: str) -> str:
    if pd.isna(q): return ""
    q = str(q)
    q = EMOJI_RE.sub(" ", q)
    q = EXTRA_SYMBOLS_RE.sub(" ", q)
    q = q.lower()
    q = re.sub(r"([!?])\1{1,}", r"\1", q)
    q = re.sub(r"\s+", " ", q).strip()
    return q

# ---- web_id / doc_id из метаданных чанков ----
def detect_doc_id_column(df: pd.DataFrame):
    for c in ["web_id", "doc_id", "original_id"]:
        if c in df.columns:
            return c
    return None

def build_docid_vector(chunks_df: pd.DataFrame) -> np.ndarray:
    col = detect_doc_id_column(chunks_df)
    if col:
        return chunks_df[col].astype("int64").to_numpy()
    if "chunk_id" in chunks_df.columns:  # <webid>_<local_idx>
        tmp = chunks_df["chunk_id"].astype(str).str.extract(r"^(\d+)_", expand=False)
        return tmp.fillna(-1).astype("int64").to_numpy()
    raise ValueError("Нет web_id/doc_id/original_id и не удаётся извлечь из chunk_id")

# ---- чтение чанков и эмбеддинги ----
def load_chunks(chunks_csv: str, text_col: str = "chunk_text") -> pd.DataFrame:
    df = pd.read_csv(chunks_csv)
    if text_col not in df.columns:
        raise ValueError(f"В {chunks_csv} нет колонки '{text_col}'. Доступные: {list(df.columns)}")
    df = df[df[text_col].astype(str).str.strip().astype(bool)].reset_index(drop=True)
    if len(df) == 0:
        raise ValueError("После фильтрации пустых текстов не осталось чанков")
    return df

def compute_embeddings(model_name: str, texts: list[str], batch_size: int = 128, add_prefix: str | None = None) -> np.ndarray:
    # add_prefix — для e5: "passage: " / "query: "
    if add_prefix:
        texts = [f"{add_prefix}{t}" for t in texts]
    model = SentenceTransformer(model_name)
    # Автоиспользование GPU, если доступен
    if hasattr(model, "to"):
        import torch
        device = "cuda" if torch.cuda.is_available() else "cpu"
        model = model.to(device)
    X = model.encode(texts, batch_size=batch_size, show_progress_bar=True, convert_to_numpy=True).astype("float32")
    return X

# ---- индекс ----
def build_faiss_index(emb: np.ndarray, metric: str = "cosine"):
    if metric == "cosine":
        faiss.normalize_L2(emb)
        index = faiss.IndexFlatIP(emb.shape[1])  # inner product == cosine на нормализованных
    else:
        index = faiss.IndexFlatL2(emb.shape[1])
    index.add(emb)
    return index

# ---- агрегация чанков в документы: сумма топ-3 скоров ----
def top_docs_from_chunks(indices_row: np.ndarray, scores_row: np.ndarray, doc_ids_vec: np.ndarray, k: int = 5) -> list[int]:
    docs = doc_ids_vec[indices_row]
    df = pd.DataFrame({"doc": docs, "s": scores_row})
    agg = (df.groupby("doc")["s"]
             .apply(lambda x: np.sort(x.values)[-3:].sum())
             .sort_values(ascending=False))
    top_docs = [int(d) for d in agg.index.tolist() if d != -1][:k]
    return top_docs

# ---- RRF fusion для списков документов ----
def rrf_fusion(dense_ranks: dict[int,int], bm25_ranks: dict[int,int], k: int = 5, K: int = 60) -> list[int]:
    docs = set(dense_ranks) | set(bm25_ranks)
    scored = []
    BIG = 10**9
    for d in docs:
        r1 = dense_ranks.get(d, BIG)
        r2 = bm25_ranks.get(d, BIG)
        s = 1.0/(K + r1) + 1.0/(K + r2)
        scored.append((d, s))
    scored.sort(key=lambda x: -x[1])
    return [d for d,_ in scored][:k]


In [ ]:
# ----- НАСТРОЙКИ -----
# Попробуй e5 — часто даёт ощутимый рост на мультиязычном корпусе:
MODEL_NAME = "intfloat/multilingual-e5-base"  # <-- было MiniLM, сменили на e5
TEXT_COL = "chunk_text"
BATCH = 256

IS_E5 = "e5" in MODEL_NAME.lower()

# ВОЗЬМЁМ улучшенный файл чанков, если сделали Cell 3.5:
# CHUNKS_CSV = "/content/chunked_websites_better.csv"  # <-- РАСКОММЕНТИРУЙ, если перечанковал
# иначе оставь твой /content/chunked_websites.csv

# 1) грузим чанки
chunks_df = load_chunks(CHUNKS_CSV, text_col=TEXT_COL)
print("Чанков:", len(chunks_df))

# 2) эмбеддинги корпуса (e5 требует префикс "passage: ")
corpus_prefix = "passage: " if IS_E5 else None
corpus_emb = compute_embeddings(MODEL_NAME, chunks_df[TEXT_COL].astype(str).tolist(),
                                batch_size=BATCH, add_prefix=corpus_prefix)

# 3) строим ЧАНКОВЫЙ FAISS
index_chunks = build_faiss_index(corpus_emb, metric="cosine")

# 4) doc_ids для каждого чанка
doc_ids_vec = build_docid_vector(chunks_df)

# 5) СТРОИМ ДОКУМЕНТНЫЙ ЭМБЕДДИНГ: среднее по чанкам документа (взвешенное по длине чанка)
lens = chunks_df.get("chunk_length_words", pd.Series(np.ones(len(chunks_df), dtype=int)))
lens = np.clip(lens.values.astype("float32"), 1.0, None)
doc_ids_unique, inv = np.unique(doc_ids_vec, return_inverse=True)
doc_sum = np.zeros((len(doc_ids_unique), corpus_emb.shape[1]), dtype="float32")
doc_w   = np.zeros((len(doc_ids_unique),), dtype="float32")
for i in range(corpus_emb.shape[0]):
    d = inv[i]
    w = lens[i]
    doc_sum[d] += corpus_emb[i] * w
    doc_w[d]   += w
doc_emb = (doc_sum / doc_w[:, None]).astype("float32")

# 6) ДОКУМЕНТНЫЙ FAISS
index_docs = build_faiss_index(doc_emb, metric="cosine")

# 7) сохраняем индексы и мета
faiss.write_index(index_chunks, "/content/faiss_index_chunks.faiss")
faiss.write_index(index_docs,   "/content/faiss_index_docs.faiss")

keep_cols = [c for c in ["chunk_id", "chunk_text", "web_id", "doc_id", "original_id"] if c in chunks_df.columns]
meta_df = chunks_df.loc[:, keep_cols].copy()
for c in ["web_id", "doc_id", "original_id"]:
    if c in meta_df.columns:
        meta_df[c] = pd.to_numeric(meta_df[c], errors="coerce").astype("Int64")
with open("/content/chunks_meta.pkl", "wb") as f:
    pickle.dump(meta_df, f)
print("Индексы (chunks & docs) и мета сохранены в /content/")


In [4]:
import re, pickle, numpy as np, pandas as pd, faiss
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

# ---- лёгкий препроцесс вопросов ----
EMOJI_RE = re.compile(
    "["                                 # emoji ranges
    "\U0001F600-\U0001F64F"
    "\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF"
    "\U0001F1E0-\U0001F1FF"
    "\U00002700-\U000027BF"
    "\U0001F900-\U0001F9FF"
    "\U00002600-\U000026FF"
    "\U00002B00-\U00002BFF"
    "\U0001FA70-\U0001FAFF"
    "]+", flags=re.UNICODE
)
EXTRA_SYMBOLS_RE = re.compile(r"[\u200d\u200c\uFE0E\uFE0F\U0001F3FB-\U0001F3FF]")

def preprocess_query(q: str) -> str:
    if pd.isna(q): return ""
    q = str(q)
    q = EMOJI_RE.sub(" ", q)
    q = EXTRA_SYMBOLS_RE.sub(" ", q)
    q = q.lower()
    q = re.sub(r"([!?])\1{1,}", r"\1", q)
    q = re.sub(r"\s+", " ", q).strip()
    return q

# ---- web_id / doc_id из метаданных чанков ----
def detect_doc_id_column(df: pd.DataFrame):
    for c in ["web_id", "doc_id", "original_id"]:
        if c in df.columns:
            return c
    return None

def build_doc_id_vector(chunks_df: pd.DataFrame) -> np.ndarray:
    col = detect_doc_id_column(chunks_df)
    if col:
        return chunks_df[col].astype("int64").to_numpy()
    if "chunk_id" in chunks_df.columns:  # <webid>_<local_idx>
        tmp = chunks_df["chunk_id"].astype(str).str.extract(r"^(\d+)_", expand=False)
        return tmp.fillna(-1).astype("int64").to_numpy()
    raise ValueError("Нет web_id/doc_id/original_id и не удаётся извлечь из chunk_id")

# ---- чтение чанков и эмбеддинги ----
def load_chunks(chunks_csv: str, text_col: str = "chunk_text") -> pd.DataFrame:
    df = pd.read_csv(chunks_csv)
    if text_col not in df.columns:
        raise ValueError(f"В {chunks_csv} нет колонки '{text_col}'. Доступные: {list(df.columns)}")
    df = df[df[text_col].astype(str).str.strip().astype(bool)].reset_index(drop=True)
    if len(df) == 0:
        raise ValueError("После фильтрации пустых текстов не осталось чанков")
    return df

def compute_embeddings(model_name: str, texts: list[str], batch_size: int = 128, add_prefix: str | None = None) -> np.ndarray:
    # add_prefix — для e5: "passage: " / "query: "
    if add_prefix:
        texts = [f"{add_prefix}{t}" for t in texts]
    model = SentenceTransformer(model_name)
    # Автоиспользование GPU, если доступен
    if hasattr(model, "to"):
        import torch
        device = "cuda" if torch.cuda.is_available() else "cpu"
        model = model.to(device)
    X = model.encode(texts, batch_size=batch_size, show_progress_bar=True, convert_to_numpy=True).astype("float32")
    return X

# ---- индекс ----
def build_faiss_index(emb: np.ndarray, metric: str = "cosine"):
    if metric == "cosine":
        faiss.normalize_L2(emb)
        index = faiss.IndexFlatIP(emb.shape[1])  # inner product == cosine на нормализованных
    else:
        index = faiss.IndexFlatL2(emb.shape[1])
    index.add(emb)
    return index

# ---- агрегация чанков в документы: сумма топ-3 скоров ----
def top_docs_from_chunks(indices_row: np.ndarray, scores_row: np.ndarray, doc_ids_vec: np.ndarray, k: int = 5) -> list[int]:
    docs = doc_ids_vec[indices_row]
    df = pd.DataFrame({"doc": docs, "s": scores_row})
    agg = (df.groupby("doc")["s"]
             .apply(lambda x: np.sort(x.values)[-3:].sum())
             .sort_values(ascending=False))
    top_docs = [int(d) for d,_ in agg.index.tolist() if d != -1][:k]
    return top_docs

# ---- RRF fusion для списков документов ----
def rrf_fusion(dense_ranks: dict[int,int], bm25_ranks: dict[int,int], k: int = 5, K: int = 60) -> list[int]:
    docs = set(dense_ranks) | set(bm25_ranks)
    scored = []
    BIG = 10**9
    for d in docs:
        r1 = dense_ranks.get(d, BIG)
        r2 = bm25_ranks.get(d, BIG)
        s = 1.0/(K + r1) + 1.0/(K + r2)
        scored.append((d, s))
    scored.sort(key=lambda x: -x[1])
    return [d for d,_ in scored][:k]



ModuleNotFoundError: No module named 'faiss'

In [ ]:
def detect_doc_id_column(df: pd.DataFrame):
    for c in ["web_id", "doc_id", "original_id"]:
        if c in df.columns:
            return c
    return None

DOC_ID_COL = detect_doc_id_column(chunks_df)
if DOC_ID_COL is None:
    # fallback: пробуем вытащить doc_id из chunk_id вида "<id>_<...>"
    tmp = chunks_df["chunk_id"].astype(str).str.extract(r"^(\d+)_", expand=False)
    chunks_df["web_id"] = pd.to_numeric(tmp, errors="coerce").astype("Int64")
    DOC_ID_COL = "web_id"

TITLE_COLS = [c for c in ["article_title","title","h1"] if c in chunks_df.columns]
title_map = {}
if TITLE_COLS:
    titles = (chunks_df.assign(_title = chunks_df[TITLE_COLS].apply(
                    lambda r: next((str(x) for x in r if pd.notna(x) and str(x).strip()), ""), axis=1))
              .groupby(DOC_ID_COL)["_title"].apply(lambda s: next((t for t in s if t), "")))
    title_map = titles.to_dict()

def build_doc_text(g):
    doc_id = g.name
    body = " ".join(map(str, g[TEXT_COL].tolist()))
    title = title_map.get(doc_id, "")
    boosted = ((title + " ") * 3 + body) if title else body
    return boosted

doc_text = (chunks_df.groupby(DOC_ID_COL).apply(build_doc_text).reset_index())
doc_text.columns = [DOC_ID_COL, TEXT_COL]

from rank_bm25 import BM25Okapi
bm25_doc_ids = doc_text[DOC_ID_COL].astype(int).to_numpy()
bm25_tokens  = [t.lower().split() for t in doc_text[TEXT_COL].tolist()]
bm25 = BM25Okapi(bm25_tokens)

# на всякий — положим идентификаторы документов «под рукой»
docs_from_index = doc_ids_unique

print(f"BM25 готов: документов {len(bm25_doc_ids)} (title-boost активен)")


# Улучшенная токенизация RU + стоп-слова
import re
STOP_RU = {
    "и","в","во","не","что","он","на","я","с","со","как","а","то","все","она","так","его","но","да",
    "ты","к","у","же","вы","за","бы","по","только","ее","мне","было","вот","от","меня","еще","нет",
    "о","из","ему","теперь","когда","даже","ну","вдруг","ли","если","уже","или","ни","быть","был",
    "него","до","вас","нибудь","опять","уж","вам","ведь","там","потом","себя","ничего","ей","может",
    "они","тут","где","есть","надо","ней","для","мы","тебя","их","чем","была","сам","чтоб","без",
    "будто","чего","раз","тоже","себе","под","будет","ж","тогда","кто","этот","того","потому","этого"
}

TOKEN_RE = re.compile(r"[a-zA-Zа-яА-ЯёЁ0-9%€$₽]+", flags=re.IGNORECASE)

def tokenize_ru(text: str):
    return [t for t in TOKEN_RE.findall(text.lower()) if t not in STOP_RU and len(t) > 1]

# перестроим BM25 с новой токенизацией
doc_tokens = [tokenize_ru(t) for t in doc_text[TEXT_COL].tolist()]
from rank_bm25 import BM25Okapi
bm25 = BM25Okapi(doc_tokens)

from collections import Counter
def top_terms_from_docs(doc_ids: list[int], topn=8):
    # соберём токены этих документов
    all_idx = [int(np.where(bm25_doc_ids == d)[0][0]) for d in doc_ids if d in set(bm25_doc_ids)]
    toks = []
    for i in all_idx:
        toks.extend(doc_tokens[i])
    # частотка с простым IDF (bm25.idf)
    scores = Counter()
    for t in toks:
        scores[t] += bm25.idf.get(t, 0.0)
    return [w for w,_ in scores.most_common(topn)]

def expand_query_with_prf(qtext: str, prf_docs: list[int], topn=8):
    terms = top_terms_from_docs(prf_docs, topn=topn)
    return qtext + " " + " ".join(terms)



In [ ]:
# === Cell 8: финальный поиск + PRF + мультиязычный кросс-энкодер-реранк (фикс OOM) ===
from sentence_transformers import CrossEncoder, SentenceTransformer
import torch, numpy as np, pandas as pd, faiss

# ---- sanity: всё ли готово из предыдущих клеток ----
req = ['index_docs','index_chunks','doc_ids_vec','doc_ids_unique',
       'bm25','bm25_doc_ids','chunks_df','TEXT_COL','IS_E5','MODEL_NAME',
       'tokenize_ru','expand_query_with_prf']
for v in req:
    assert v in globals(), f"{v} не найден — проверь предыдущие клетки."

# ---- параметры ----
K = 5
RETRIEVE_K = 250
DOCS_FOR_RERANK = 30        # сколько документов отправляем в реранкер
CHUNKS_PER_DOC_RERANK = 2   # сколько чанков на документ в реранк
RERANK_AGG = "max"          # 'max' или 'top2sum'
Krrf = 60
BIG = 10**9

# ---- вопросы ----
qdf = pd.read_csv(QUESTIONS_CSV)
assert "q_id" in qdf.columns, "В questions.csv должна быть колонка q_id"
q_col = "query" if "query" in qdf.columns else qdf.columns[1]
qdf["__clean"] = qdf[q_col].apply(preprocess_query)
queries = qdf["__clean"].astype(str).tolist()

# ---- энкодеры (грузим один раз) ----
# 1) Query-энкодер: держим на CPU (дёшево для 1 строки и экономим GPU для реранкера)
query_encoder = SentenceTransformer(MODEL_NAME, device="cpu")
query_prefix = "query: " if IS_E5 else None

# 2) Реранкер на GPU
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", device="cuda", max_length=256)
reranker.model.eval()

def ranks_from_list(doc_list):
    return {int(d): r for r, d in enumerate(doc_list)}

# заранее: тексты чанков
chunk_texts = chunks_df[TEXT_COL].tolist()

rows = []
with torch.inference_mode():
    for q_i, qid in enumerate(qdf["q_id"].tolist()):
        qtext_raw = queries[q_i]

        # ---- PRF: берём топ-10 BM25 по сырому запросу, расширяем запрос топ-термами
        bm_scores_raw = bm25.get_scores(tokenize_ru(qtext_raw))
        idx_sorted_raw = np.argsort(bm_scores_raw)[::-1][:10]
        seed_docs = bm25_doc_ids[idx_sorted_raw].tolist()
        qtext = expand_query_with_prf(qtext_raw, seed_docs, topn=8)

        # ---- эмбеддинг ДЛЯ ОДНОГО расширенного запроса (CPU, одна строка)
        q_inp = [f"{query_prefix}{qtext}" if query_prefix else qtext]
        q_emb = query_encoder.encode(q_inp, batch_size=1, convert_to_numpy=True).astype("float32")
        faiss.normalize_L2(q_emb)

        # ---- поиски: документный и чанковый индексы
        D_docs,   I_docs   = index_docs.search(q_emb, RETrIEVE_K:=RETRIEVE_K)
        D_chunks, I_chunks = index_chunks.search(q_emb, RETrIEVE_K)

        docs_from_index = doc_ids_unique

        # ---- doc-index ранги
        docs_from_docindex = [int(docs_from_index[j]) for j in I_docs[0]]
        doc_ranks = ranks_from_list(docs_from_docindex)

        # ---- chunk-index → агрегируем топ-3 скоров чанков в документы
        def top_docs_from_chunks(indices_row, scores_row, doc_ids_vec, k=RETRIEVE_K):
            docs = doc_ids_vec[indices_row]
            df = pd.DataFrame({"doc": docs, "s": scores_row})
            agg = (df.groupby("doc")["s"]
                     .apply(lambda x: np.sort(x.values)[-3:].sum())
                     .sort_values(ascending=False))
            return [int(d) for d in agg.index.tolist() if d != -1][:k]

        dense_docs_from_chunks = top_docs_from_chunks(I_chunks[0], D_chunks[0], doc_ids_vec, k=RETRIEVE_K)
        dense_chunk_ranks = ranks_from_list(dense_docs_from_chunks)

        # ---- BM25 ранги (по расширенному запросу)
        bm_scores = bm25.get_scores(tokenize_ru(qtext))
        idx_sorted = np.argsort(bm_scores)[::-1][:RETRIEVE_K]
        bm_docs = bm25_doc_ids[idx_sorted]
        bm25_ranks = {int(d): r for r, d in enumerate(bm_docs)}

        # ---- RRF → кандидаты на реранк
        docs_all = set(doc_ranks) | set(dense_chunk_ranks) | set(bm25_ranks)
        scored = []
        for d in docs_all:
            r1 = doc_ranks.get(d, BIG); r2 = dense_chunk_ranks.get(d, BIG); r3 = bm25_ranks.get(d, BIG)
            scored.append((d, 1/(Krrf + r1) + 1/(Krrf + r2) + 1/(Krrf + r3)))
        scored.sort(key=lambda x: -x[1])
        cand_docs = [d for d,_ in scored[:DOCS_FOR_RERANK]]

        # ---- топ-чанки каждого кандидата из чанкового поиска
        idx_row, scr_row = I_chunks[0], D_chunks[0]
        doc2chunks = {}
        for chunk_idx, sc in zip(idx_row, scr_row):
            d = int(doc_ids_vec[chunk_idx])
            if d in cand_docs:
                lst = doc2chunks.setdefault(d, [])
                if len(lst) < CHUNKS_PER_DOC_RERANK or sc > lst[-1][1]:
                    lst.append((chunk_idx, float(sc)))
                    lst.sort(key=lambda x: -x[1])
                    del lst[CHUNKS_PER_DOC_RERANK:]

        # ---- пары (query, chunk) → реранк (GPU)
        pairs, pair_doc = [], []
        for d, ch_list in doc2chunks.items():
            for (ch_idx, _) in ch_list:
                pairs.append((qtext, chunk_texts[ch_idx]))
                pair_doc.append(d)

        if pairs:
            scores = reranker.predict(pairs, batch_size=128, show_progress_bar=False, convert_to_numpy=True)
            doc_best = {}
            for (d, sc) in zip(pair_doc, scores):
                doc_best.setdefault(d, []).append(float(sc))
            reranked = []
            for d, lst in doc_best.items():
                lst.sort(reverse=True)
                agg = lst[0] if RERANK_AGG == "max" or len(lst) == 1 else (lst[0] + lst[1])
                reranked.append((d, agg))
            reranked.sort(key=lambda x: -x[1])
            final_docs = [d for d,_ in reranked[:K]]
        else:
            final_docs = cand_docs[:K]

        rows.append({"q_id": int(qid), "web_list": str(final_docs)})

# ---- сохраняем submit ----
submit = pd.DataFrame(rows)
submit_path = "/content/submit.csv"
submit.to_csv(submit_path, index=False, encoding="utf-8")
submit.head(), submit_path


In [ ]:
from google.colab import files
files.download("/content/submit.csv")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 12.9 MB/s eta 0:00:00
Презентация создана: RAG_Deep_Dive.pptx
